# 🚀 Meilenstein 6: Production-Ready Hyperparameter Tuning mit Optuna

## Lernziele:

### Modularisierung: Wir brechen das monolithische Skript in saubere, wiederverwendbare Funktionen auf (Data Pipeline, Model Factory, Training Loop).
Best Practices: Device-Agnostizität (CPU/GPU/Apple MPS), Seed-Management für Reproduzierbarkeit.

### Optuna Integration: Automatisierte Suche mit Pruning (intelligenter Abbruch schlechter Trials).
Final Retraining: Das "Best-Practice"-Pattern: Optuna findet das Rezept, danach backen wir den finalen Kuchen (Retraining auf den besten Parametern).

In [ ]:
%pip install plotly

In [ ]:
%pip install optuna

In [1]:
# Imports & Reproduzierbarkeit
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import numpy as np
import pandas as pd
import joblib
import copy
import optuna
from optuna.visualization import plot_optimization_history, plot_param_importances

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# --- BEST PRACTICE: Reproduzierbarkeit garantieren ---
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# --- BEST PRACTICE: Device Agnostizität (Unterstützt NVIDIA, Apple Silicon & CPU) ---
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps") # Für MacBooks mit M1/M2/M3 Chips
else:
    device = torch.device("cpu")

print(f"🖥️ Training auf Device: {device}")

/Users/thomasjorg/Documents/05_VSCode/05_Python/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🖥️ Training auf Device: mps


## Schritt 1: Die Data Pipeline entkoppeln
In Produktionsskripten wird die Datenaufbereitung in eine Funktion ausgelagert. Das verhindert Data Leakage und macht den Code testbar.

In [2]:
def get_iris_dataloaders(batch_size=12, test_size=0.2, val_size=0.2):
    """Lädt, bereinigt, skaliert und verpackt die Daten in PyTorch DataLoader."""
    df = pd.read_csv('Datasets/Iris.csv')
    X = df.drop('species', axis=1).values
    y = df['species'].values
    
    le = LabelEncoder()
    y = le.fit_transform(y)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42)
    X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=val_size, random_state=0)
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    
    # Konvertierung zu Tensoren und Verschiebung auf das korrekte Device
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
    test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
    
    return train_loader, val_loader, test_loader, scaler, le

# Einmaliger Aufruf für unsere Session
train_loader, val_loader, test_loader, scaler, le = get_iris_dataloaders(batch_size=12)

## Schritt 2: Die Model Factory
Anstatt das Netz hart zu codieren, bauen wir eine "Fabrik", die uns basierend auf Hyperparametern das passende Netz zurückgibt.

In [3]:
def create_model(input_dim=4, hidden_dim=16, output_dim=3, dropout_rate=0.0):
    """Erstellt ein dynamisches neuronales Netz basierend auf Hyperparametern."""
    layers = [
        nn.Linear(input_dim, hidden_dim),
        nn.ReLU(),
    ]
    if dropout_rate > 0:
        layers.append(nn.Dropout(dropout_rate))
        
    layers.append(nn.Linear(hidden_dim, output_dim))
    
    model = nn.Sequential(*layers)
    return model.to(device) # Wichtig: Modell direkt auf die GPU/MPS schieben

## Schritt 3: Der Training Loop & Optuna Objective

### Pruning vs. Early Stopping: 
Wir nutzen beides. Early Stopping beendet ein einzelnes Training, wenn es stagniert. Optunas Pruning bricht Trials ab, 
die im Vergleich zu anderen Trials hoffnungslos unterlegen sind (spart massiv Rechenzeit).

In [4]:
def objective(trial, train_loader, val_loader, max_epochs=100):
    """Die Objective-Funktion, die von Optuna für jeden Trial aufgerufen wird."""
    
    # 1. Hyperparameter Search Space definieren
    hidden_dim = trial.suggest_int("hidden_dim", 4, 64, step=4)
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5, step=0.1)
    patience = trial.suggest_int("patience", 5, 20)
    
    # 2. Modell & Optimierer instanziieren
    model = create_model(hidden_dim=hidden_dim, dropout_rate=dropout_rate)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    patience_counter = 0
    
    # 3. Training Loop
    for epoch in range(max_epochs):
        # --- Training ---
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()
            
        # --- Validation ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                val_loss += criterion(model(X_batch), y_batch).item()
                
        val_loss /= len(val_loader)
        
        # --- Optuna Pruning (Der Gamechanger) ---
        # Wir berichten Optuna den aktuellen Fortschritt
        trial.report(val_loss, epoch)
        # Wenn der Trial im Vergleich zu anderen Trials zu schlecht ist -> Abbruch!
        if trial.should_prune():
            raise optuna.TrialPruned()
            
        # --- Klassisches Early Stopping (Trial-Intern) ---
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            
        if patience_counter >= patience:
            break # Trial ist konvergiert/stagniert
            
    return best_val_loss

## Schritt 4: Die Studie konfigurieren und starten
Wir nutzen den MedianPruner. Dieser schaut sich die_mediane Performance aller bisherigen Trials an. Liegt unser aktueller Trial unter dem Median, wird er gnadenlos gestoppt.

In [5]:
# Study anlegen
study = optuna.create_study(
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=42), # Tree-structured Parzen Estimator (Bayesian Optimization)
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5)
)

# Studie starten (Lambda-Funktion nutzt sich die DataLoader aus dem Scope)
print("🚀 Starte Hyperparameter-Suche...")
study.optimize(
    lambda trial: objective(trial, train_loader, val_loader), 
    n_trials=30, # In der Praxis oft 50-100, für Iris reichen 30 zum Testen
    show_progress_bar=True
)

print("\n🏆 Bester Trial:")
print(f"  Value (Val Loss): {study.best_value:.4f}")
print(f"  Params: {study.best_params}")

[I 2026-05-26 10:40:08,023] A new study created in memory with name: no-name-d12afd43-c11e-4ddd-b4e3-2e933f182108


🚀 Starte Hyperparameter-Suche...


Best trial: 0. Best value: 0.156522:   3%|▎         | 1/30 [00:00<00:25,  1.14it/s]

[I 2026-05-26 10:40:08,908] Trial 0 finished with value: 0.1565215215086937 and parameters: {'hidden_dim': 24, 'lr': 0.0711447600934342, 'weight_decay': 0.0008471801418819979, 'dropout_rate': 0.30000000000000004, 'patience': 7}. Best is trial 0 with value: 0.1565215215086937.


Best trial: 0. Best value: 0.156522:   7%|▋         | 2/30 [00:02<00:30,  1.09s/it]

...

[I 2026-05-26 10:40:20,846] Trial 25 finished with value: 0.14487630408257246 and parameters: {'hidden_dim': 44, 'lr': 0.025791288364019876, 'weight_decay': 4.006866426965164e-05, 'dropout_rate': 0.0, 'patience': 18}. Best is trial 11 with value: 0.09391302894800901.
[I 2026-05-26 10:40:20,946] Trial 26 pruned. 
[I 2026-05-26 10:40:21,031] Trial 27 pruned. 


Best trial: 11. Best value: 0.093913:  97%|█████████▋| 29/30 [00:13<00:00,  3.54it/s]

[I 2026-05-26 10:40:21,566] Trial 28 finished with value: 0.14961925148963928 and parameters: {'hidden_dim': 60, 'lr': 0.05133639699962025, 'weight_decay': 0.005009427132460238, 'dropout_rate': 0.0, 'patience': 19}. Best is trial 11 with value: 0.09391302894800901.


Best trial: 11. Best value: 0.093913: 100%|██████████| 30/30 [00:13<00:00,  2.17it/s]

[I 2026-05-26 10:40:21,873] Trial 29 finished with value: 0.140042326413095 and parameters: {'hidden_dim': 40, 'lr': 0.09160693807170987, 'weight_decay': 0.00036648323747145284, 'dropout_rate': 0.4, 'patience': 16}. Best is trial 11 with value: 0.09391302894800901.

🏆 Bester Trial:
  Value (Val Loss): 0.0939
  Params: {'hidden_dim': 60, 'lr': 0.0898092329347876, 'weight_decay': 0.008303103281084483, 'dropout_rate': 0.0, 'patience': 16}


## Schritt 5: Visualisierung der Suche
Optuna liefert out-of-the-box Production-Ready Plots für Dashboards (z.B. Streamlit) oder Notebooks.

In [6]:
# Zeigt, wie sich der beste Loss über die Zeit entwickelt hat
fig1 = plot_optimization_history(study)
fig1.show()

# Zeigt, welche Hyperparameter den größten Einfluss auf den Loss hatten
fig2 = plot_param_importances(study)
fig2.show()

## Schritt 6: Das "Final Retraining" Pattern 🎓
### Wichtiges Konzept für die Produktion:
Optuna speichert standardmäßig nicht die Gewichte aller 30 Modelle (das würde den RAM sprengen). Es speichert nur die Parameter.
Der Best-Practice-Weg ist: Wir nehmen das beste "Rezept" (Parameter) und trainieren das Modell ein letztes Mal, um die finalen Gewichte für das Deployment zu speichern.

In [7]:
def train_final_model(best_params, train_loader, val_loader, max_epochs=200):
    """Trainiert das finale Modell mit den besten Parametern und gibt die Gewichte zurück."""
    set_seed(42)
    model = create_model(hidden_dim=best_params['hidden_dim'], dropout_rate=best_params['dropout_rate'])
    optimizer = optim.AdamW(model.parameters(), lr=best_params['lr'], weight_decay=best_params['weight_decay'])
    criterion = nn.CrossEntropyLoss()
    
    best_val_loss = float('inf')
    best_weights = None
    patience_counter = 0
    
    for epoch in range(max_epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                val_loss += criterion(model(X_batch), y_batch).item()
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_weights = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 30: # Großzügige Patience für das finale Training
                break
                
    return model, best_weights

# Finale Ausführung
print("🏗️ Trainiere finales Modell mit den besten Parametern...")
final_model, final_weights = train_final_model(study.best_params, train_loader, val_loader)

# Evaluation auf dem echten Test-Set
final_model.load_state_dict(final_weights)
final_model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = final_model(X_batch)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(y_batch.numpy())

test_acc = accuracy_score(all_targets, all_preds)
print(f"✅ Finale Testgenauigkeit: {test_acc:.2%}")

# Deployment-Export
torch.save(final_weights, 'Models/iris_net_optuna_best.pth')
joblib.dump(study.best_params, 'Models/optuna_best_params.pkl') # Rezept mitspeichern!
joblib.dump(le, 'Models/label_encoder.pkl')
joblib.dump(scaler, 'Models/scaler.pkl')
print("💾 Modell und Artefakte erfolgreich exportiert.")

🏗️ Trainiere finales Modell mit den besten Parametern...
✅ Finale Testgenauigkeit: 100.00%
💾 Modell und Artefakte erfolgreich exportiert.
